In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import pandas as pd

In [16]:
from cyclical_MCMC import load_data
from Heston import full_trunc_euler, moment_vec, moment_labels, SMM, print_SMM

# 1. Load Data

In [4]:
Y, dY, dY_pd, Y_indx, _ = load_data('data/SP500.csv')

Data from 2016-09-19 00:00:00 to 2026-09-17 00:00:00


# 2. Setup Estimation with Simulated Method of Moments

### Settings for simulation

In [5]:
years = 10
days_pr_year = 252
T_SMM = years*days_pr_year #len(Y) # total number of days
burnin = 2*days_pr_year # two years for burnin
N = 1000
Delta = 1

### Empirical Moments

In [6]:
K_y = 4
L = 10
K_rv = 10
K_yrv = 5

rv_cap = 30

dY_capped = dY[dY**2 < rv_cap]
print(Y_indx[1:][dY**2 > 40])
M_data_noextremes = moment_vec(dY_capped, K_y, L, K_rv, K_yrv)
M_data = moment_vec(dY, K_y, L, K_rv, K_yrv)

DatetimeIndex(['2020-03-09', '2020-03-12', '2020-03-13', '2020-03-16',
               '2020-03-24', '2020-04-06', '2025-04-09'],
              dtype='datetime64[ns]', name='observation_date', freq=None)


In [15]:
tbl = pd.DataFrame({"All data": M_data, "No extremes": M_data_noextremes},
                   index=moment_labels(K_y, L, K_rv, K_yrv))

tbl

,All data,No extremes
E[y],0.050665,0.049802
E|y|^1,0.735306,0.703151
E|y|^2,1.304972,1.007310
E|y|^3,4.971814,2.220471
E|y|^4,33.318681,6.526621
E|y_t y_t-1|,0.824992,0.618682
E|y_t y_t-2|,0.853951,0.643189
E|y_t y_t-3|,0.831084,0.649170
E|y_t y_t-4|,0.824605,0.637469
E|y_t y_t-5|,0.799148,0.622028


# Estimate SMM with all data

In [8]:
seed = 42
N_multi = 20
iter = 20
W_tol = 1e-8

# bounds for optimization
lb = [0, 0, 0.1, 0.005, -0.99]
ub = [0.2, 0.1, 2.5, 0.15, 0.99]

In [9]:
lens_roll = N_multi*[10]

# bounds for drawing initial guesses
bounds = {
    'mu_l' : 0.02, 'mu_h' : 0.07,
    'kappa_l' : 0.01, 'kappa_h' : 0.1,
    'theta_l' : 0.5, 'theta_h' : 2,
    'sigma_sq_V_l' : 0.01, 'sigma_sq_V_h' : 0.06,
    'rho_l' : -0.8, 'rho_h' : -0.2
}

In [10]:
SMM_alldata = SMM(dY, N_multi, bounds, lb, ub, K_y, L, K_rv, K_yrv, seed, Delta, N, T_SMM, burnin)

----------------------------------------------------------------------------------------------------
Starting Multistart i = 0 with guess:
mu = 0.03873
kappa = 0.06507
theta = 0.68306
sigma_sq_V = 0.02943
rho = -0.28214
Ended after 17 iterations with status = True
Theta:
mu = 0.06366
kappa = 0.02611
theta = 0.86869
sigma_sq_V = 0.03873
rho = -0.99000
cost = 0.00555307

----------------------------------------------------------------------------------------------------
----------------------------------------------------------------------------------------------------
Starting Multistart i = 1 with guess:
mu = 0.06754
kappa = 0.02255
theta = 1.24277
sigma_sq_V = 0.02357
rho = -0.42602
Ended after 18 iterations with status = True
Theta:
mu = 0.06367
kappa = 0.02605
theta = 0.86843
sigma_sq_V = 0.03864
rho = -0.99000
cost = 0.00555310

----------------------------------------------------------------------------------------------------
------------------------------------------------------

In [20]:
print_SMM(SMM_alldata)

              estimate      s.e.  95% CI lower  95% CI upper
mu             0.06366   0.01665       0.03103       0.09629
kappa          0.02611   0.02251      -0.01801       0.07022
theta          0.86869   0.06057       0.74996       0.98741
sigma_sq_V     0.03873   0.03356      -0.02704       0.10451
rho           -0.99000   0.35723      -1.69016      -0.28984


# Estimate SMM without extreme Realized Variance

In [11]:
SMM_noextremes = SMM(dY_capped, N_multi, bounds, lb, ub, K_y, L, K_rv, K_yrv, seed, Delta, N, T_SMM, burnin)

----------------------------------------------------------------------------------------------------
Starting Multistart i = 0 with guess:
mu = 0.03873
kappa = 0.06507
theta = 0.68306
sigma_sq_V = 0.02943
rho = -0.28214
Ended after 22 iterations with status = True
Theta:
mu = 0.06920
kappa = 0.03005
theta = 0.79817
sigma_sq_V = 0.03786
rho = -0.89253
cost = 0.00835995

----------------------------------------------------------------------------------------------------
----------------------------------------------------------------------------------------------------
Starting Multistart i = 1 with guess:
mu = 0.06754
kappa = 0.02255
theta = 1.24277
sigma_sq_V = 0.02357
rho = -0.42602
Ended after 19 iterations with status = True
Theta:
mu = 0.06920
kappa = 0.03005
theta = 0.79816
sigma_sq_V = 0.03786
rho = -0.89214
cost = 0.00835994

----------------------------------------------------------------------------------------------------
------------------------------------------------------

In [19]:
print_SMM(SMM_noextremes)

              estimate      s.e.  95% CI lower  95% CI upper
mu             0.06920   0.01582       0.03820       0.10020
kappa          0.03006   0.02511      -0.01915       0.07928
theta          0.79817   0.05191       0.69643       0.89990
sigma_sq_V     0.03787   0.03138      -0.02362       0.09937
rho           -0.89203   0.32738      -1.53370      -0.25037
